In [8]:
import polars as pl

auth = pl.scan_parquet("filtered_auth.parquet")

# Keep only real users
auth_users = (
    auth
    .filter(pl.col("src_user").str.starts_with("U"))
    .rename({"src_user": "user"})
)

# Build user-level features
user_features = (
    auth_users
    .group_by("user")
    .agg([
        pl.len().alias("total_logins"),
        pl.col("status").filter(pl.col("status") == "Failure").count().alias("failed_logins"),
        pl.col("dst_computer").n_unique().alias("unique_destinations")
    ])
)


In [9]:
red = pl.scan_csv(
    "data_windows/redteam.txt.gz",
    has_header=False,
    new_columns=["time", "user", "src_computer", "dst_computer"]
)

comp_users = (
    red
    .group_by("user")
    .agg([
        pl.len().alias("compromise_events")
    ])
)


In [10]:
user_stats = (
    user_features
    .join(comp_users, on="user", how="left")
    .with_columns(
        pl.col("compromise_events").fill_null(0),
    )
)


In [12]:
user_stats_df = user_stats.collect(streaming=True)

print(user_stats_df.shape)
print(user_stats_df.sort("compromise_events", descending=True).head())


C:\Users\kruti\AppData\Local\Temp\ipykernel_3968\3787688524.py:1: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  user_stats_df = user_stats.collect(streaming=True)


(25951, 5)
shape: (5, 5)
┌────────────┬──────────────┬───────────────┬─────────────────────┬───────────────────┐
│ user       ┆ total_logins ┆ failed_logins ┆ unique_destinations ┆ compromise_events │
│ ---        ┆ ---          ┆ ---           ┆ ---                 ┆ ---               │
│ str        ┆ u32          ┆ u32           ┆ u32                 ┆ u32               │
╞════════════╪══════════════╪═══════════════╪═════════════════════╪═══════════════════╡
│ U66@DOM1   ┆ 4977072      ┆ 0             ┆ 238                 ┆ 118               │
│ U3005@DOM1 ┆ 18068        ┆ 0             ┆ 66                  ┆ 36                │
│ U737@DOM1  ┆ 27203        ┆ 0             ┆ 129                 ┆ 32                │
│ U1653@DOM1 ┆ 88210        ┆ 0             ┆ 3362                ┆ 31                │
│ U293@DOM1  ┆ 100265       ┆ 0             ┆ 78                  ┆ 31                │
└────────────┴──────────────┴───────────────┴─────────────────────┴───────────────────┘


In [13]:
user_features = (
    auth_users
    .group_by("user")
    .agg([
        pl.len().alias("total_logins"),
        pl.col("status").filter(pl.col("status") == "Failure").count().alias("failed_logins"),
        pl.col("dst_computer").n_unique().alias("unique_destinations"),
        pl.col("src_computer").n_unique().alias("unique_sources"),
    ])
)


In [14]:
user_stats = user_stats.with_columns(
    (pl.col("failed_logins") / pl.col("total_logins")).alias("failure_ratio")
)


In [15]:
import pandas as pd

df = user_stats_df.to_pandas()

# Features only
features = df[[
    "total_logins",
    "failed_logins",
    "unique_destinations"
]]

labels = (df["compromise_events"] > 0).astype(int)


In [ ]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(
    n_estimators=200,
    contamination=0.01,   # assume ~1% anomalies
    random_state=42
)

iso.fit(features)

df["anomaly_score"] = iso.decision_function(features)
df["anomaly_flag"] = iso.predict(features)


In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(labels, -df["anomaly_score"])
print("ROC-AUC:", auc)


ROC-AUC: 0.8824685054953557


In [14]:
df_sorted = df.sort_values("anomaly_score")

print(df_sorted.head(20)[[
    "user",
    "anomaly_score",
    "compromise_events"
]])


             user  anomaly_score  compromise_events
2796     U66@DOM1      -0.148048                118
20131    U78@DOM1      -0.132115                  2
14453     U8@DOM1      -0.124970                  0
12375    U24@DOM1      -0.117885                  5
17870   U194@DOM1      -0.112541                  0
3727    U726@DOM1      -0.109741                  0
3602     U13@DOM1      -0.105561                  2
16287  U1653@DOM1      -0.101677                 31
1755     U94@DOM1      -0.098582                  0
19290  U7056@DOM1      -0.098364                  0
8662    U316@DOM1      -0.095887                  0
24543   U293@DOM1      -0.095338                 31
23909    U14@DOM1      -0.094401                  0
16261    U48@DOM1      -0.093852                  0
5025     U19@DOM1      -0.093852                  0
198      U22@DOM1      -0.093657                  0
14256    U34@DOM1      -0.093031                  0
20314   U346@DOM1      -0.092210                  0
2709    U292

In [15]:
# Top 50 most anomalous users
top_50 = df.sort_values("anomaly_score").head(50)

print("Compromised in Top 50:", top_50["compromise_events"].gt(0).sum())


Compromised in Top 50: 7


In [ ]:
# How many compromised total?
print("Total compromised users:", labels.sum())


Total compromised users: 104


In [17]:
import numpy as np

df_eval = df.copy()

# Binary label
df_eval["is_compromised"] = (df_eval["compromise_events"] > 0).astype(int)

def precision_at_k(df, k):
    top_k = df.sort_values("anomaly_score").head(k)
    return top_k["is_compromised"].sum() / k

for k in [10, 20, 50, 100, 200]:
    print(f"Precision@{k}: {precision_at_k(df_eval, k):.4f}")


Precision@10: 0.5000
Precision@20: 0.3000
Precision@50: 0.1400
Precision@100: 0.1300
Precision@200: 0.1400


In [18]:
total_compromised = df_eval["is_compromised"].sum()

def recall_at_k(df, k):
    top_k = df.sort_values("anomaly_score").head(k)
    return top_k["is_compromised"].sum() / total_compromised

for k in [10, 20, 50, 100, 200]:
    print(f"Recall@{k}: {recall_at_k(df_eval, k):.4f}")


Recall@10: 0.0481
Recall@20: 0.0577
Recall@50: 0.0673
Recall@100: 0.1250
Recall@200: 0.2692


In [19]:
df_eval.sort_values("anomaly_score").head(20)[[
    "user",
    "total_logins",
    "unique_destinations",
    "failed_logins",
    "compromise_events",
    "anomaly_score"
]]


,user,total_logins,unique_destinations,failed_logins,compromise_events,anomaly_score
2796,U66@DOM1,4977072,238,0,118,-0.148048
20131,U78@DOM1,600352,79,0,2,-0.132115
14453,U8@DOM1,653163,61,0,0,-0.124970
12375,U24@DOM1,484545,57,0,5,-0.117885
17870,U194@DOM1,154754,68,0,0,-0.112541
3727,U726@DOM1,170173,62,0,0,-0.109741
3602,U13@DOM1,518241,47,0,2,-0.105561
16287,U1653@DOM1,88210,3362,0,31,-0.101677
1755,U94@DOM1,1211187,3,0,0,-0.098582
19290,U7056@DOM1,220349,50,0,0,-0.098364


UPDATE FEATURES TO IMPROVE PERFORMANCE

In [54]:
import numpy as np
from sklearn.ensemble import IsolationForest

df = user_stats_df.to_pandas()

# -----------------------------
# 🔥 Keep Strong Scale Features
# -----------------------------

# Ratio features
df["failure_ratio"] = df["failed_logins"] / df["total_logins"]
df["dest_ratio"] = df["unique_destinations"] / df["total_logins"]

# Avoid divide-by-zero
df = df.fillna(0)

# -----------------------------
# 📊 Feature Matrix
# -----------------------------

features = df[[
    "total_logins",
    "unique_destinations",
    "failed_logins",
    "failure_ratio",
    "dest_ratio"
]]

labels = (df["compromise_events"] > 0).astype(int)
df["is_compromised"] = labels

# -----------------------------
# 🌲 Isolation Forest
# -----------------------------

iso = IsolationForest(
    n_estimators=400,
    contamination=0.01,
    random_state=42
)

iso.fit(features)

df["anomaly_score"] = iso.decision_function(features)
df["anomaly_flag"] = iso.predict(features)


In [61]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(labels, -df["anomaly_score"])
print("Upgraded Behavioral ROC-AUC:", auc)

def precision_at_k(df, k):
    top_k = df.sort_values("anomaly_score").head(k)
    return top_k["is_compromised"].sum() / k

for k in [10, 20, 50, 100]:
    print(f"Precision@{k}: {precision_at_k(df, k):.4f}")


Upgraded Behavioral ROC-AUC: 0.8258503813863236
Precision@10: 0.5000
Precision@20: 0.3500
Precision@50: 0.1400
Precision@100: 0.1100


ADDING HE PIPELINE

In [62]:
df_behavior = df_users.copy()
df_behavior["behavioral_score"] = df["anomaly_score"]

# Convert to percentile risk
df_behavior["behavioral_risk"] = (
    df_behavior["behavioral_score"]
    .rank(pct=True)
)


In [63]:
df_behavior["behavioral_risk"] = 1 - df_behavior["behavioral_risk"]


In [69]:
df_behavior = df_behavior.merge(
    df_eval[["user", "temporal_score"]],
    on="user",
    how="left"
)

df_behavior["temporal_risk"] = (
    df_behavior["temporal_score"]
    .rank(pct=True)
)

df_behavior["temporal_risk"] = 1 - df_behavior["temporal_risk"]


In [70]:
df_behavior["risk_score"] = (
    0.7 * df_behavior["behavioral_risk"] +
    0.3 * df_behavior["temporal_risk"]
)


In [71]:
df_behavior["risk_score"] *= 100


In [72]:
top_alerts = df_behavior.sort_values("risk_score", ascending=False).head(20)

top_alerts[[
    "user",
    "risk_score",
    "total_logins",
    "unique_destinations",
    "compromise_events"
]]


,user,risk_score,total_logins,unique_destinations,compromise_events
1675,U66@DOM1,99.994991,4977072,238,118
11090,U8@DOM1,99.988440,653163,61,0
13483,U78@DOM1,99.984201,600352,79,2
16489,U1653@DOM1,99.974182,88210,3362,31
22907,U24@DOM1,99.974182,484545,57,5
11208,U7056@DOM1,99.963778,220349,50,0
3322,U13@DOM1,99.961659,518241,47,2
771,U292@DOM1,99.951447,1930319,35,0
19166,U346@DOM1,99.950676,1812931,11,0
12210,U22@DOM1,99.940850,8198287,30,0


In [73]:
def generate_incident_report(user_row):
    report = f"""
    Incident Report
    ----------------
    User: {user_row['user']}
    Risk Score: {user_row['risk_score']:.2f}

    Behavioral Indicators:
    - Total Logins: {user_row['total_logins']}
    - Unique Machines Accessed: {user_row['unique_destinations']}
    - Failed Logins: {user_row['failed_logins']}

    Assessment:
    User exhibits anomalous login behavior relative to enterprise baseline.
    Elevated machine spread suggests potential lateral movement.

    Recommended Actions:
    - Temporarily disable account
    - Force credential reset
    - Audit accessed hosts
    """
    return report


In [75]:
# Sort by highest risk
top_alert = df_behavior.sort_values(
    "risk_score", ascending=False
).iloc[0]

print("Top flagged user:")
print(top_alert[[
    "user",
    "risk_score",
    "total_logins",
    "unique_destinations",
    "failed_logins"
]])


Top flagged user:
user                    U66@DOM1
risk_score             99.994991
total_logins             4977072
unique_destinations          238
failed_logins                  0
Name: 1675, dtype: object


In [76]:
incident_text = generate_incident_report(top_alert)

print(incident_text)



    Incident Report
    ----------------
    User: U66@DOM1
    Risk Score: 99.99

    Behavioral Indicators:
    - Total Logins: 4977072
    - Unique Machines Accessed: 238
    - Failed Logins: 0

    Assessment:
    User exhibits anomalous login behavior relative to enterprise baseline.
    Elevated machine spread suggests potential lateral movement.

    Recommended Actions:
    - Temporarily disable account
    - Force credential reset
    - Audit accessed hosts
    


TEMPORAL MODELLING LAYER

In [1]:
import polars as pl

# Load filtered auth lazily
auth = pl.scan_parquet("filtered_auth.parquet")

# Keep only real users
auth_users = (
    auth
    .filter(pl.col("src_user").str.starts_with("U"))
    .rename({"src_user": "user"})
)

# Define window size
WINDOW_SIZE = 10000   # adjust later if needed

# Add window id column
auth_windowed = auth_users.with_columns(
    (pl.col("time") // WINDOW_SIZE).alias("window_id")
)


In [2]:
window_features = (
    auth_windowed
    .group_by(["user", "window_id"])
    .agg([
        pl.len().alias("login_count"),
        pl.col("dst_computer").n_unique().alias("unique_destinations"),
        pl.col("status").filter(pl.col("status") == "Failure").count().alias("failure_count")
    ])
)


In [3]:
window_df = window_features.collect(streaming=True)

print(window_df.shape)
window_df.head()


C:\Users\kruti\AppData\Local\Temp\ipykernel_3968\332608811.py:1: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  window_df = window_features.collect(streaming=True)


(1485841, 5)


user,window_id,login_count,unique_destinations,failure_count
str,i64,u32,u32,u32
"""U2993@DOM1""",234,33,7,0
"""U9403@DOM1""",238,90,5,0
"""U9303@DOM1""",160,7,2,0
"""U3569@DOM1""",185,47,14,0
"""U874@DOM1""",194,93,10,0


In [4]:
import numpy as np
from sklearn.ensemble import IsolationForest

df_w = window_df.to_pandas()

# Add ratio feature
df_w["dest_ratio"] = df_w["unique_destinations"] / df_w["login_count"]
df_w["failure_ratio"] = df_w["failure_count"] / df_w["login_count"]

df_w = df_w.fillna(0)

features = df_w[[
    "login_count",
    "unique_destinations",
    "failure_count",
    "dest_ratio",
    "failure_ratio"
]]

iso = IsolationForest(
    n_estimators=300,
    contamination=0.01,
    random_state=42
)

iso.fit(features)

df_w["anomaly_score"] = iso.decision_function(features)
df_w["anomaly_flag"] = iso.predict(features)


In [64]:
user_window_scores = (
    df_w.groupby("user")["anomaly_score"]
    .min()
    .reset_index()
)

user_window_scores = user_window_scores.rename(
    columns={"anomaly_score": "temporal_score"}
)


In [65]:
df_users = user_stats_df.to_pandas()

df_users["is_compromised"] = (df_users["compromise_events"] > 0).astype(int)

df_eval = df_users.merge(user_window_scores, on="user", how="left")

from sklearn.metrics import roc_auc_score

auc_temporal = roc_auc_score(
    df_eval["is_compromised"],
    -df_eval["temporal_score"]
)

print("Temporal ROC-AUC:", auc_temporal)


Temporal ROC-AUC: 0.7358267288868519


In [68]:
def precision_at_k(df, k):
    top_k = df.sort_values("temporal_score").head(k)
    return top_k["is_compromised"].sum() / k

for k in [10, 20, 50, 100]:
    print(f"Temporal Precision@{k}: {precision_at_k(df_eval, k):.4f}")


Temporal Precision@10: 0.3000
Temporal Precision@20: 0.3000
Temporal Precision@50: 0.1800
Temporal Precision@100: 0.1100


MERGE SCORES

In [19]:
# Behavioral scores (best baseline version)
df_behavioral = df_users.copy()
df_behavioral["behavioral_score"] = df_w["anomaly_score"]

# Temporal scores already in df_eval
df_combined = df_eval.copy()

# Merge behavioral score
df_combined = df_combined.merge(
    df_behavioral[["user", "behavioral_score"]],
    on="user",
    how="left"
)


In [23]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# Rank users (lower score = more anomalous)
df_combined["behavioral_rank"] = df_combined["behavioral_score"].rank()
df_combined["temporal_rank"] = df_combined["temporal_score"].rank()

# Combine ranks (lower rank = more suspicious)
df_combined["ensemble_rank"] = (
    df_combined["behavioral_rank"] +
    df_combined["temporal_rank"]
)

# For evaluation, smaller rank = more anomalous
df_combined = df_combined.sort_values("ensemble_rank")



In [24]:
from sklearn.metrics import roc_auc_score

# Convert rank to anomaly score (invert so higher = worse)
df_combined["ensemble_score"] = -df_combined["ensemble_rank"]

auc_ensemble = roc_auc_score(
    df_combined["is_compromised"],
    df_combined["ensemble_score"]
)

print("Rank Ensemble ROC-AUC:", auc_ensemble)

def precision_at_k(df, k):
    top_k = df.head(k)
    return top_k["is_compromised"].sum() / k

for k in [10, 20, 50, 100]:
    print(f"Rank Ensemble Precision@{k}: {precision_at_k(df_combined, k):.4f}")


Rank Ensemble ROC-AUC: 0.6527089142914964
Rank Ensemble Precision@10: 0.0000
Rank Ensemble Precision@20: 0.0500
Rank Ensemble Precision@50: 0.1000
Rank Ensemble Precision@100: 0.0500


PER-USER DEVIATION MODEL

In [25]:
# Compute per-user mean and std
user_baseline = df_w.groupby("user").agg({
    "login_count": ["mean", "std"],
    "unique_destinations": ["mean", "std"],
    "failure_count": ["mean", "std"]
})

# Flatten column names
user_baseline.columns = [
    "login_mean", "login_std",
    "dest_mean", "dest_std",
    "fail_mean", "fail_std"
]

user_baseline = user_baseline.reset_index()

# Merge back to window data
df_dev = df_w.merge(user_baseline, on="user", how="left")


In [26]:
eps = 1e-6

df_dev["login_z"] = (df_dev["login_count"] - df_dev["login_mean"]) / (df_dev["login_std"] + eps)
df_dev["dest_z"] = (df_dev["unique_destinations"] - df_dev["dest_mean"]) / (df_dev["dest_std"] + eps)
df_dev["fail_z"] = (df_dev["failure_count"] - df_dev["fail_mean"]) / (df_dev["fail_std"] + eps)


In [27]:
df_dev["window_deviation"] = df_dev[[
    "login_z",
    "dest_z",
    "fail_z"
]].abs().max(axis=1)


In [28]:
user_deviation = (
    df_dev.groupby("user")["window_deviation"]
    .max()
    .reset_index()
)

user_deviation = user_deviation.rename(
    columns={"window_deviation": "deviation_score"}
)


In [30]:
df_eval["deviation_score"].isna().sum()


np.int64(4595)

In [31]:
df_eval["deviation_score"] = df_eval["deviation_score"].fillna(0)


In [32]:
from sklearn.metrics import roc_auc_score

auc_dev = roc_auc_score(
    df_eval["is_compromised"],
    df_eval["deviation_score"]
)

print("Per-User Deviation ROC-AUC:", auc_dev)


Per-User Deviation ROC-AUC: 0.7702885843022995


In [33]:
def precision_at_k(df, k):
    top_k = df.sort_values("deviation_score", ascending=False).head(k)
    return top_k["is_compromised"].sum() / k

for k in [10, 20, 50, 100]:
    print(f"Deviation Precision@{k}: {precision_at_k(df_eval, k):.4f}")


Deviation Precision@10: 0.0000
Deviation Precision@20: 0.0500
Deviation Precision@50: 0.0400
Deviation Precision@100: 0.0300


LOCAL OUTLIER FACTOR

In [36]:
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import roc_auc_score

features = df[[
    "total_logins",
    "unique_destinations",
    "failed_logins"
]]

labels = (df["compromise_events"] > 0).astype(int)

# LOF (novelty=True to allow scoring)
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.01,
    novelty=True
)

lof.fit(features)

lof_scores = -lof.decision_function(features)  # invert (higher = more anomalous)

auc_lof = roc_auc_score(labels, lof_scores)
print("LOF ROC-AUC:", auc_lof)


c:\Users\kruti\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


LOF ROC-AUC: 0.5899927755341344


One-Class SVM

In [37]:
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

ocsvm = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.01
)

ocsvm.fit(features_scaled)

svm_scores = -ocsvm.decision_function(features_scaled)

auc_svm = roc_auc_score(labels, svm_scores)
print("One-Class SVM ROC-AUC:", auc_svm)


One-Class SVM ROC-AUC: 0.4761566585617733


PCA Reconstruction Error

In [38]:
from sklearn.decomposition import PCA
import numpy as np

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

pca = PCA(n_components=2)
pca.fit(features_scaled)

reconstructed = pca.inverse_transform(pca.transform(features_scaled))

reconstruction_error = np.mean((features_scaled - reconstructed) ** 2, axis=1)

auc_pca = roc_auc_score(labels, reconstruction_error)
print("PCA Reconstruction ROC-AUC:", auc_pca)


PCA Reconstruction ROC-AUC: 0.8427765385657017


LSTM AUTOENCODER

In [39]:
import numpy as np

df_seq = df_w.copy()

# Add ratios
df_seq["dest_ratio"] = df_seq["unique_destinations"] / df_seq["login_count"]
df_seq["failure_ratio"] = df_seq["failure_count"] / df_seq["login_count"]

df_seq = df_seq.fillna(0)

# Sort properly
df_seq = df_seq.sort_values(["user", "window_id"])
from sklearn.preprocessing import StandardScaler

feature_cols = [
    "login_count",
    "unique_destinations",
    "failure_count",
    "dest_ratio",
    "failure_ratio"
]

scaler = StandardScaler()
df_seq[feature_cols] = scaler.fit_transform(df_seq[feature_cols])


In [40]:
SEQUENCE_LENGTH = 10


In [41]:
SEQUENCE_LENGTH = 10

user_sequences = []
user_labels = []

users = df_seq["user"].unique()

for user in users:
    user_data = df_seq[df_seq["user"] == user]
    values = user_data[feature_cols].values
    
    if len(values) >= SEQUENCE_LENGTH:
        for i in range(len(values) - SEQUENCE_LENGTH + 1):
            seq = values[i:i+SEQUENCE_LENGTH]
            user_sequences.append(seq)
            user_labels.append(user)


In [43]:
X = np.array(user_sequences)

print("Sequence shape:", X.shape)

Sequence shape: (1327756, 10, 5)


In [47]:
import torch
import torch.nn as nn

class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        
        self.encoder = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.decoder = nn.LSTM(hidden_dim, input_dim, batch_first=True)
    
    def forward(self, x):
        _, (hidden, _) = self.encoder(x)
        hidden = hidden.repeat(x.size(1), 1, 1).permute(1,0,2)
        reconstructed, _ = self.decoder(hidden)
        return reconstructed

input_dim = X.shape[2]
hidden_dim = 32

model = LSTMAutoencoder(input_dim, hidden_dim)


In [48]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

X_tensor = torch.tensor(X, dtype=torch.float32)

EPOCHS = 20
BATCH_SIZE = 128

for epoch in range(EPOCHS):
    perm = torch.randperm(X_tensor.size(0))
    
    for i in range(0, X_tensor.size(0), BATCH_SIZE):
        indices = perm[i:i+BATCH_SIZE]
        batch = X_tensor[indices]
        
        optimizer.zero_grad()
        reconstructed = model(batch)
        loss = criterion(reconstructed, batch)
        loss.backward()
        optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.6f}")


Epoch 1, Loss: 0.020822
Epoch 2, Loss: 0.067098
Epoch 3, Loss: 0.273196
Epoch 4, Loss: 0.020970
Epoch 5, Loss: 0.109411
Epoch 6, Loss: 0.051006
Epoch 7, Loss: 0.017731
Epoch 8, Loss: 0.264449
Epoch 9, Loss: 0.304235
Epoch 10, Loss: 0.058378
Epoch 11, Loss: 0.097920
Epoch 12, Loss: 0.116050
Epoch 13, Loss: 0.071901
Epoch 14, Loss: 0.023133
Epoch 15, Loss: 0.071349
Epoch 16, Loss: 0.039570
Epoch 17, Loss: 0.047217
Epoch 18, Loss: 0.020656
Epoch 19, Loss: 0.149128
Epoch 20, Loss: 0.014403


In [49]:
model.eval()

errors_list = []

with torch.no_grad():
    for i in range(0, X_tensor.size(0), BATCH_SIZE):
        batch = X_tensor[i:i+BATCH_SIZE]
        reconstructed = model(batch)
        batch_errors = torch.mean((reconstructed - batch) ** 2, dim=(1,2))
        errors_list.append(batch_errors)

errors = torch.cat(errors_list).numpy()


In [50]:
import pandas as pd

seq_df = pd.DataFrame({
    "user": user_labels,
    "error": errors
})

user_error = seq_df.groupby("user")["error"].max().reset_index()

df_users = user_stats_df.to_pandas()
df_users["is_compromised"] = (df_users["compromise_events"] > 0).astype(int)

df_eval = df_users.merge(user_error, on="user", how="left")
df_eval["error"] = df_eval["error"].fillna(0)

from sklearn.metrics import roc_auc_score

auc_lstm = roc_auc_score(df_eval["is_compromised"], df_eval["error"])
print("LSTM Autoencoder ROC-AUC:", auc_lstm)


LSTM Autoencoder ROC-AUC: 0.7044966533833714


In [51]:
def precision_at_k(df, k):
    top_k = df.sort_values("error", ascending=False).head(k)
    return top_k["is_compromised"].sum() / k

for k in [10, 20, 50, 100]:
    print(f"LSTM Precision@{k}: {precision_at_k(df_eval, k):.4f}")


LSTM Precision@10: 0.2000
LSTM Precision@20: 0.1000
LSTM Precision@50: 0.1200
LSTM Precision@100: 0.0600
